In [0]:
%sql
SELECT * FROM `workspace`.`default`.`credit_card_fraud`;

transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
1,84.47,22,Electronics,0,0,66,3,40,0
2,541.82,3,Travel,1,0,87,1,64,0
3,237.01,17,Grocery,0,0,49,1,61,0
4,164.33,4,Grocery,0,1,72,3,34,0
5,30.53,15,Food,0,0,79,0,44,0
6,30.53,13,Clothing,0,0,90,2,46,0
7,10.77,18,Travel,0,0,48,1,28,0
8,362.02,13,Electronics,0,0,68,1,40,0
9,165.43,8,Grocery,0,0,80,0,21,0
10,221.63,5,Grocery,0,0,59,1,34,0


In [0]:
df = _sqldf.dropna()

In [0]:
fraud_df = df.filter("is_fraud = 1")


because I have $5 limited tokens in open ai


In [0]:
sample = fraud_df.limit(10).toPandas()
data_text = sample.to_string(index=False)

In [0]:
%pip install openai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


because dataset has some weird column names

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "amount",
    when(df.amount < 50, "Low")
    .when(df.amount < 200, "Medium")
    .otherwise("High")
)

In [0]:
df = df.withColumn(
    "transaction_hour",
    (df.transaction_hour / 3600).cast("int")  # hour bucket
)

In [0]:
from openai import OpenAI
client = OpenAI(api_key="API_KEY")

prompt = f"""
You are a senior fraud detection analyst.

Do NOT repeat the dataset.
Do NOT display the table.

For each transaction, provide analysis in this exact format:

Transaction ID: <id>
- Reason: <why it is suspicious>
- Risk Level: <Low/Medium/High>
- Suggested Action: <what should be done>

Be concise and business-focused.

Data:
{data_text}
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=150
)

print(response.choices[0].message.content)

Transaction ID: 58
- Reason: High-risk category and foreign transaction with location mismatch.
- Risk Level: High
- Suggested Action: Block the transaction and investigate further.

Transaction ID: 122
- Reason: Large amount for grocery purchase combined with foreign transaction.
- Risk Level: High
- Suggested Action: Block the transaction and initiate a review of the account.

Transaction ID: 124
- Reason: Small amount but foreign transaction and location mismatch suggest fraud.
- Risk Level: Medium
- Suggested Action: Flag for further monitoring and reach out to the cardholder for verification.

Transaction ID: 289
- Reason: Clothing purchase with foreign transaction and high-risk profile.
- Risk Level: High
- Suggested Action: Block


**ML Prediction starts**

In [0]:
df = spark.table("transactions")

# Basic cleaning
df = df.dropna()

In [0]:
df = df.select(
    "amount",
    "transaction_hour",
    "merchant_category",
    "foreign_transaction",
    "location_mismatch",
    "device_trust_score",
    "velocity_last_24h",
    "cardholder_age",
    "is_fraud"
)

In [0]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="merchant_category",
    outputCol="merchant_category_index"
)

df = indexer.fit(df).transform(df)


In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "amount",
        "transaction_hour",
        "merchant_category_index",
        "foreign_transaction",
        "location_mismatch",
        "device_trust_score",
        "velocity_last_24h",
        "cardholder_age"
    ],
    outputCol="features"
)

df = assembler.transform(df)

In [0]:
train, test = df.randomSplit([0.8, 0.2], seed=42)

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="is_fraud"
)

model = lr.fit(train)

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_fraud",
    numTrees=50
)

rf_model = rf.fit(train)
rf_predictions = rf_model.transform(test)

In [0]:
from pyspark.sql.functions import col

predictions = predictions.withColumn(
    "risk_score",
    col("probability")[1] * 100
)

In [0]:
predictions = model.transform(test)

predictions.select(
    "is_fraud",
    "prediction",
    "probability"
).show(10)

+--------+----------+--------------------+
|is_fraud|prediction|         probability|
+--------+----------+--------------------+
|       0|       0.0|[0.99999970497499...|
|       0|       0.0|[0.99999985149832...|
|       0|       0.0|[0.99979843897148...|
|       0|       0.0|[0.99999885464878...|
|       0|       0.0|[0.99545469370631...|
|       0|       0.0|[0.99999949812802...|
|       0|       0.0|[0.99999966604687...|
|       0|       0.0|[0.99999749233966...|
|       0|       0.0|[0.99998734123156...|
|       0|       0.0|[0.99999988621994...|
+--------+----------+--------------------+
only showing top 10 rows


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud"
)

auc = evaluator.evaluate(predictions)
print("AUC:", auc)

AUC: 0.9876021745081937


In [0]:
predictions.filter("prediction = 1").show()

+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+-----------------------+--------------------+--------------------+--------------------+----------+
|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|merchant_category_index|            features|       rawPrediction|         probability|prediction|
+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+-----------------------+--------------------+--------------------+--------------------+----------+
| 11.22|               1|          Grocery|                  0|                1|                25|                4|            32|       1|                    3.0|[11.22,1.0,3.0,0....|[-3.8328189765539...|[0.02118977598347...|       1.0|
| 19.66|               1|      Elect

In [0]:
predictions.groupBy("is_fraud", "prediction").count().show()

+--------+----------+-----+
|is_fraud|prediction|count|
+--------+----------+-----+
|       1|       1.0|   12|
|       1|       0.0|   15|
|       0|       1.0|    5|
|       0|       0.0| 1889|
+--------+----------+-----+



%md
Big Problem: missing fraud cases
Total fraud cases = 27 (12 + 15)
detected only 12
 Detection rate (Recall):

12 / 27 ≈ 44%


In [0]:
model.setThreshold(0.3)

LogisticRegressionModel: uid=LogisticRegression_9fa01da4a8a9, numClasses=2, numFeatures=8

In [0]:
fraud = df.filter("is_fraud = 1")
non_fraud = df.filter("is_fraud = 0").sample(0.2)

df_balanced = fraud.union(non_fraud)

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

recall_eval = MulticlassClassificationEvaluator(
    labelCol="is_fraud",
    metricName="recallByLabel"
)

print("Recall:", recall_eval.evaluate(predictions))

Recall: 0.9973600844772967


In [0]:
MulticlassClassificationEvaluator(metricName="recallByLabel")

MulticlassClassificationEvaluator_cccb7769f519

In [0]:
recall_eval = MulticlassClassificationEvaluator(
    labelCol="is_fraud",
    metricName="recallByLabel",
    metricLabel=1
)

print("Fraud Recall:", recall_eval.evaluate(predictions))

Fraud Recall: 0.4444444444444444


CALL AI for Insights

In [0]:
suspicious_df = predictions.filter("prediction = 1").limit(5)

In [0]:
pdf = suspicious_df.toPandas()

In [0]:
from openai import OpenAI
client = OpenAI(api_key="API_KEY")

In [0]:
def create_prompt(row):
    return f"""
You are a fraud detection analyst.

Analyze this transaction:

Amount: {row['amount']}
Velocity (24h): {row['velocity_last_24h']}
Device Trust Score: {row['device_trust_score']}
Location Mismatch: {row['location_mismatch']}
Prediction Probability: {row['probability'][1]}

Explain:
1. Why this might be fraud
2. Risk level (Low/Medium/High)
3. Suggested action for bank

Be concise and business-focused.
"""

In [0]:
responses = []

for _, row in pdf.iterrows():
    prompt = create_prompt(row)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # low cost
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150
    )
    
    responses.append(response.choices[0].message.content)

In [0]:
pdf["ai_explanation"] = responses
for i, text in enumerate(pdf["ai_explanation"]):
    print(f"\n--- Transaction {i} ---")
    print(text)



--- Transaction 0 ---
1. **Why this might be fraud:**
   - The transaction amount of $11.22 is relatively low, which can be a tactic used by fraudsters to test stolen payment methods.
   - A velocity of 4 within 24 hours indicates a higher frequency of transactions, which can be a red flag for unusual behavior.
   - A low Device Trust Score of 25 suggests that the device used for the transaction has been flagged or is considered less secure, indicating potential risk.
   - Location Mismatch indicates that the transaction is coming from a geographic location inconsistent with the account holder's usual location, raising suspicion about the legitimacy of the transaction.
   - The high Prediction Probability of 0.9788 suggests that the model strongly predicts that

--- Transaction 1 ---
1. **Why this might be fraud:**
   - **Velocity:** A velocity of 5 transactions in 24 hours is relatively high for typical consumer behavior, which may indicate suspicious activity.
   - **Device Trust Sc

In [0]:
# Extract risk scores from probability column
if 'probability' in pdf.columns:
    pdf['risk_score'] = pdf['probability'].apply(lambda x: x[1] * 100)
    display(pdf[['amount', 'is_fraud', 'prediction', 'risk_score']].head())
else:
    print("Probability column not found in dataframe")

amount,is_fraud,prediction,risk_score
11.22,1,1.0,97.88102240165269
19.66,1,1.0,86.88819776609817
21.1,1,1.0,52.71800364366831
33.34,1,1.0,99.71872520375281
50.78,0,1.0,53.74413665763663


In [0]:
# high-risk transactions with business logic
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import when, col, concat_ws, lit, array_remove, array, concat

fresh_predictions = model.transform(test)

#risk score
high_risk_df = fresh_predictions.withColumn(
    "risk_score",
    (vector_to_array("probability").getItem(1) * 100).cast("int")
)

# risk level
high_risk_df = high_risk_df.withColumn(
    "risk_level",
    when(col("risk_score") >= 80, "High")
    .when(col("risk_score") >= 50, "Medium")
    .otherwise("Low")
)

#  action
high_risk_df = high_risk_df.withColumn(
    "action",
    when(col("risk_score") >= 80, "Block")
    .when(col("risk_score") >= 50, "Manual Review")
    .otherwise("Allow")
)

# detailed explanation based on actual features
from pyspark.sql.functions import coalesce

high_risk_df = high_risk_df.withColumn(
    "explanation",
    concat_ws(" + ",
        array(
            coalesce(when(col("risk_score") >= 70, lit("High risk score")), lit("")),
            coalesce(when(col("device_trust_score") < 6,
                          concat(lit("New device (trust="), col("device_trust_score"), lit(")"))), lit("")),
            coalesce(when(col("location_mismatch") == 1, lit("Location mismatch")), lit("")),
            coalesce(when(col("foreign_transaction") == 1, lit("Foreign transaction")), lit("")),
            coalesce(when(col("velocity_last_24h") >= 3,
                          concat(lit("High velocity ("), col("velocity_last_24h"), lit(" txns)"))), lit(""))
        )
    )
)
    

# Filter and show high-risk transactions
high_risk_df.filter("risk_score >= 50").limit(5).select(
    "amount", "risk_score", "risk_level", "action", "explanation"
).show(truncate=False)

+------+----------+----------+-------------+-------------------------------------------------------------------------------------+
|amount|risk_score|risk_level|action       |explanation                                                                          |
+------+----------+----------+-------------+-------------------------------------------------------------------------------------+
|11.22 |97        |High      |Block        |High risk score +  + Location mismatch +  + High velocity (4 txns)                   |
|19.66 |86        |High      |Block        |High risk score +  +  + Foreign transaction + High velocity (5 txns)                 |
|21.1  |52        |Medium    |Manual Review| +  + Location mismatch +  +                                                         |
|33.34 |99        |High      |Block        |High risk score +  + Location mismatch + Foreign transaction + High velocity (4 txns)|
|50.78 |53        |Medium    |Manual Review| +  +  + Foreign transaction + High vel

In [0]:
# Regenerate predictions to avoid corrupted DataFrame
fresh_predictions = model.transform(test)
fresh_predictions.groupBy("is_fraud", "prediction").count().show()

+--------+----------+-----+
|is_fraud|prediction|count|
+--------+----------+-----+
|       1|       1.0|   12|
|       1|       0.0|   15|
|       0|       1.0|    5|
|       0|       0.0| 1889|
+--------+----------+-----+



In [0]:
from pyspark.sql.functions import sum

tp = fresh_predictions.filter("is_fraud = 1 AND prediction = 1").count()
fn =  fresh_predictions.filter("is_fraud = 1 AND prediction = 0").count()

recall = tp / (tp + fn)
print("Fraud Recall:", recall)

Fraud Recall: 0.4444444444444444


In [0]:
model.setThreshold(0.3)

LogisticRegressionModel: uid=LogisticRegression_24672b9cd89c, numClasses=2, numFeatures=8

In [0]:
fresh_predictions = model.transform(test)

In [0]:
final_df = high_risk_df.select(
    "amount",
    "transaction_hour",
    "merchant_category",
    "risk_score",
    "risk_level",
    "action",
    "explanation"
)
display(final_df)


amount,transaction_hour,merchant_category,risk_score,risk_level,action,explanation
0.01,17,Travel,0,Low,Allow,+ + + + High velocity (4 txns)
0.04,13,Electronics,0,Low,Allow,+ + + +
0.07,5,Travel,0,Low,Allow,+ + + +
0.14,2,Travel,0,Low,Allow,+ + + +
0.26,4,Food,0,Low,Allow,+ + + + High velocity (3 txns)
0.42,22,Grocery,0,Low,Allow,+ + + +
0.67,2,Food,0,Low,Allow,+ + + +
0.79,12,Travel,0,Low,Allow,+ + + +
1.0,18,Travel,0,Low,Allow,+ + Location mismatch + +
1.02,19,Food,0,Low,Allow,+ + + +
